In [1]:
import os

USE_TF = False
print("Using TensorFlow:", USE_TF)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["OMP_NUM_THREADS"] = "4"

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
import joblib

from PIL import Image

import numpy as np
import pandas as pd
import zipfile
import re

if USE_TF:
    import tensorflow as tf

    tf.random.set_seed(42)




Using TensorFlow: False


In [2]:
IMG_SIZE = 150
BATCH_SIZE = 32
EPOCHS = 1
VALIDATION_SPLIT = 0.2
MODEL_NUM = 1
TRAIN_ZIP = "/kaggle/input/dogs-vs-cats-redux-kernels-edition/train.zip"
TEST_ZIP = "/kaggle/input/dogs-vs-cats-redux-kernels-edition/test.zip"
WORKING_DIR = "/kaggle/working"




In [3]:
train_root_path = os.path.join(
    WORKING_DIR, "dogs-vs-cats-redux-kernels-edition", "train"
)
if not os.path.isdir(train_root_path):
    with zipfile.ZipFile(TRAIN_ZIP, "r") as z:
        z.extractall(WORKING_DIR)

TRAIN_ROOT = train_root_path
assert os.path.isdir(TRAIN_ROOT), f"Training directory not found at {TRAIN_ROOT}"




In [4]:
cat_dir = os.path.join(TRAIN_ROOT, "cat")
dog_dir = os.path.join(TRAIN_ROOT, "dog")
assert os.path.isdir(cat_dir) and os.path.isdir(dog_dir), "Cat/Dog subfolders missing."

if USE_TF:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator

    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        validation_split=VALIDATION_SPLIT,
    )

    train_generator = train_datagen.flow_from_directory(
        TRAIN_ROOT,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="binary",
        subset="training",
        shuffle=True,
        seed=42,
    )

    val_generator = train_datagen.flow_from_directory(
        TRAIN_ROOT,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="binary",
        subset="validation",
        shuffle=False,
        seed=42,
    )


def se_block(input_tensor, reduction=16):
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Dense(filters // reduction, activation="relu")(se)
    se = layers.Dense(filters, activation="sigmoid")(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([input_tensor, se])


def build_model():
    """Create the Keras model – only used when TensorFlow is available."""
    from tensorflow.keras import layers, models, regularizers
    import tensorflow as tf

    model = models.Sequential(
        [
            layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
            layers.Conv2D(
                32,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.001),
            ),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2, 2),
            layers.Conv2D(
                64,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.001),
            ),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2, 2),
            layers.Conv2D(
                128,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.001),
            ),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2, 2),
            layers.Conv2D(
                256,
                (3, 3),
                padding="same",
                activation="relu",
                kernel_regularizer=regularizers.l2(0.001),
            ),
            layers.BatchNormalization(),
            layers.MaxPooling2D(2, 2),
            layers.GlobalAveragePooling2D(),
            layers.BatchNormalization(),
            layers.Dense(
                512, activation="relu", kernel_regularizer=regularizers.l2(0.001)
            ),
            layers.Dropout(0.5),
            layers.Dense(
                256, activation="relu", kernel_regularizer=regularizers.l2(0.001)
            ),
            layers.Dropout(0.4),
            layers.Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        optimizer="adam",
        metrics=["accuracy"],
    )
    return model


def train_sklearn_model():
    """Fallback training using scikit‑learn on down‑sampled images with scaling."""
    print("Loading training images for sklearn model...")

    def load_one(path):
        img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        return np.asarray(img, dtype=np.uint8).ravel()

    cat_files = [
        os.path.join(cat_dir, f)
        for f in sorted(os.listdir(cat_dir))
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]
    dog_files = [
        os.path.join(dog_dir, f)
        for f in sorted(os.listdir(dog_dir))
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    all_files = cat_files + dog_files
    labels = np.array([0] * len(cat_files) + [1] * len(dog_files), dtype=np.int8)

    # use as many cores as reasonable (up to 8) for parallel image loading
    n_jobs = min(8, os.cpu_count() or 1)
    from joblib import Parallel, delayed

    X = np.stack(
        Parallel(n_jobs=n_jobs, prefer="processes")(
            delayed(load_one)(p) for p in all_files
        ),
        axis=0,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X, labels, test_size=VALIDATION_SPLIT, random_state=42, stratify=labels
    )
    pipeline = Pipeline(
        [
            # avoid extra copies and keep data in float32 to halve memory bandwidth
            ("scaler", StandardScaler(copy=False, dtype=np.float32)),
            (
                "clf",
                LogisticRegression(
                    max_iter=1000,
                    n_jobs=n_jobs,  # use limited cores
                    solver="saga",
                    random_state=42,
                    C=1.0,
                ),
            ),
        ]
    )
    pipeline.fit(X_train, y_train)

    val_acc = pipeline.score(X_val, y_val)
    print(f"Validation accuracy (sklearn pipeline): {val_acc:.4f}")
    return pipeline, (X_val, y_val)




In [5]:
if USE_TF:
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    callbacks = [
        EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
    ]
else:
    callbacks = []  # No callbacks needed for sklearn fallback

if USE_TF:
    for i in range(MODEL_NUM):
        print(f"Training model {i}")
        model = build_model()
        model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=EPOCHS,
            callbacks=callbacks,
            verbose=2,
            use_multiprocessing=True,
        )
        model_path = os.path.join(WORKING_DIR, f"model{i}.h5")
        model.save(model_path)
else:
    clf, (X_val, y_val) = train_sklearn_model()
    model_path = os.path.join(WORKING_DIR, "sklearn_model.pkl")
    joblib.dump(clf, model_path)




Loading training images for sklearn model...


TypeError: StandardScaler.__init__() got an unexpected keyword argument 'dtype'

In [6]:
if USE_TF:
    sum_pred = None
    for i in range(MODEL_NUM):
        model_path = os.path.join(WORKING_DIR, f"model{i}.h5")
        model = tf.keras.models.load_model(model_path)
        pred = model.predict(val_generator, verbose=0)
        sum_pred = pred if sum_pred is None else sum_pred + pred

    avg_pred = sum_pred / MODEL_NUM
    y_pred = (avg_pred > 0.5).astype(int).ravel()
    y_true = val_generator.classes
    acc = accuracy_score(y_true, y_pred)
    print(f"Validation accuracy: {acc:.4f}")
else:
    val_proba = clf.predict_proba(X_val)[:, 1]
    val_logloss = log_loss(y_val, val_proba, eps=1e-7)
    print(f"Validation log loss (sklearn): {val_logloss:.5f}")




NameError: name 'clf' is not defined

In [7]:
test_root_path = os.path.join(WORKING_DIR, "dogs-vs-cats-redux-kernels-edition", "test")
if not os.path.isdir(test_root_path):
    with zipfile.ZipFile(TEST_ZIP, "r") as z:
        z.extractall(WORKING_DIR)

TEST_ROOT = test_root_path
assert os.path.isdir(TEST_ROOT), f"Test directory not found at {TEST_ROOT}"

test_files = []
for root, _, files in os.walk(TEST_ROOT):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            test_files.append(os.path.join(root, f))


def extract_number(fname):
    m = re.search(r"\d+", fname)
    return int(m.group()) if m else -1


sorted_files = sorted(test_files, key=lambda p: extract_number(os.path.basename(p)))
image_paths = sorted_files


def load_one_test(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.uint8).ravel()


n_jobs = min(8, os.cpu_count() or 1)

from joblib import Parallel, delayed

test_arrays = Parallel(n_jobs=n_jobs, prefer="processes")(
    delayed(load_one_test)(p) for p in image_paths
)
batch = np.stack(test_arrays, axis=0)

if USE_TF:
    sum_pred = None
    for i in range(MODEL_NUM):
        model_path = os.path.join(WORKING_DIR, f"model{i}.h5")
        model = tf.keras.models.load_model(model_path)
        pred = model.predict(batch, verbose=0)
        sum_pred = pred if sum_pred is None else sum_pred + pred

    avg_pred = sum_pred / MODEL_NUM
    labels = np.clip(avg_pred, 1e-6, 1 - 1e-6).ravel()
else:
    clf = joblib.load(model_path)
    X_test = batch  # already flattened, scaler inside pipeline will handle conversion
    probs = clf.predict_proba(X_test)[:, 1]
    labels = np.clip(probs, 1e-6, 1 - 1e-6)

ids = [os.path.splitext(os.path.basename(p))[0] for p in image_paths]

submission = pd.DataFrame({"id": ids, "label": labels})
submission_path = os.path.join(WORKING_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

NameError: name 'model_path' is not defined